# 03. Metadata Generation

이 노트북은 Track B의 metadata card를 생성합니다.

- 입력 case: `outputs/context_rag/audit_cases.csv`
- retrieval 예시: `outputs/context_rag/retrieval_index.csv`
- prompt: `prompts/03_metadata_generator_system.txt`
- schema: `schemas/03_metadata_schema.json`

각 case에 대해 target expression의 표면 의미, social context 필요 여부, normalization guidance를 JSONL로 저장합니다. 기본값은 API 비용과 출력 형식을 먼저 확인하기 위한 `LIMIT = 20` smoke run입니다.

출력:

- `outputs/context_rag/metadata_cards.jsonl`


## metadata 생성 설정

OpenAI model과 smoke run limit을 설정합니다. 전체 실행 전에는 `LIMIT = 20`을 유지합니다. 이 노트북은 `generate_metadata()`를 직접 호출하며, 같은 작업은 `scripts/03_generate_metadata.py`로 command 실행할 수 있습니다.


In [ ]:
import pathlib, runpy

BOOTSTRAP = pathlib.Path("scripts/01_02_03_04_05_06_notebook_bootstrap.py")
if not BOOTSTRAP.exists():
    BOOTSTRAP = pathlib.Path("/content/lexnorm_submit/scripts/01_02_03_04_05_06_notebook_bootstrap.py")

setup_project = runpy.run_path(str(BOOTSTRAP))["setup_project"]
PROJECT_ROOT = setup_project()

from lexnorm.utils import sync_to_drive

from lexnorm.rag import generate_metadata

MODEL = "gpt-4.1-mini"
LIMIT = 20
CASES_CSV = "outputs/context_rag/audit_cases.csv"
INDEX_CSV = "outputs/context_rag/retrieval_index.csv"
METADATA_JSONL = "outputs/context_rag/metadata_cards.jsonl"
METADATA_PROMPT = pathlib.Path("prompts/03_metadata_generator_system.txt")
METADATA_SCHEMA = pathlib.Path("schemas/03_metadata_schema.json")


## metadata card 생성 실행

Audit case와 retrieval index를 넣어 metadata JSONL을 생성합니다. OpenAI API 호출이므로 `OPENAI_API_KEY`가 필요합니다.


In [ ]:
metadata_df = generate_metadata(
    cases_csv=CASES_CSV,
    output_jsonl=METADATA_JSONL,
    prompt_path=METADATA_PROMPT,
    schema_path=METADATA_SCHEMA,
    model=MODEL,
    index_csv=INDEX_CSV,
    limit=LIMIT,
    include_gold=False,
)
print("saved", METADATA_JSONL, "new rows", len(metadata_df))
display(metadata_df.head())
sync_to_drive(METADATA_JSONL)
